# ⛏️ 02 · 工作量证明深度剖析

> nBits 编码、难度计算、预期哈希次数、泊松过程模型。

## 2.1 nBits → Target 编码

比特币区块头用 32 位 `nBits` 存储难度目标。这是压缩的浮点格式：

```
target = mantissa × 256^(exponent - 3)
```

创世区块 nBits = `0x1d00ffff`:
- exponent = 0x1d = 29
- mantissa = 0x00ffff
- target = 0x00ffff × 256^26

这是一个**非常非常大**的数——大致 2^224。

In [ ]:
def bits_to_target(bits: int) -> int:
    exp = bits >> 24
    mant = bits & 0x00ffffff
    return mant * (256 ** (exp - 3))

def target_to_bits(target: int) -> int:
    tb = target.to_bytes(32, "big")
    for i, b in enumerate(tb):
        if b != 0:
            exp = 32 - i
            break
    else:
        return 0
    mant = target // (256 ** (exp - 3))
    if mant >= 0x800000:
        mant >>= 8
        exp += 1
    return (exp << 24) | mant

genesis_bits = 0x1d00ffff
genesis_target = bits_to_target(genesis_bits)

print(f"Genesis nBits:   0x{genesis_bits:08x}")
print(f"Genesis target:  0x{genesis_target:064x}")
print(f"Round-trip:      {target_to_bits(genesis_target) == genesis_bits}")

# Current Bitcoin (2024 typical)
current_bits = 0x17035e1a
current_target = bits_to_target(current_bits)
print(f"
Current nBits:   0x{current_bits:08x}")
print(f"Current target:  0x{current_target:064x}")
print(f"Target shrunk by: {genesis_target / current_target:,.0f}x since genesis")


## 2.2 难度 (Difficulty)

难度是一个**相对值**，以创世区块的目标值为基准：

```
difficulty = genesis_target / current_target
```

当前难度约 **70 万亿**。意味着现在挖一个块的难度是 2009 年的 70 万亿倍。

> 注意：比特币代码中"难度"和"目标值"是倒数关系。难度越高 → 目标值越小 → 越难挖。

In [ ]:
def target_to_difficulty(target: int) -> float:
    return bits_to_target(0x1d00ffff) / target

def difficulty_to_target(diff: float) -> int:
    return int(bits_to_target(0x1d00ffff) / diff)

# Demonstrate a range of difficulties
for diff_label, diff in [
    ("Genesis", 1),
    ("Early GPU era", 10_000),
    ("ASIC era start", 500_000_000),
    ("2017 bull run", 1_000_000_000_000),
    ("Current (~70T)", 70_000_000_000_000),
]:
    t = difficulty_to_target(diff)
    leading_zeros = 64 - len(hex(t)[2:])
    print(f"  Difficulty {diff:>20,}: target has {leading_zeros} leading zero hex digits")


## 2.3 预期哈希次数

```
P(hit) = target / 2^256
E[tries] = 1 / P(hit) = 2^256 / target
```

这是挖矿的"工作量"的数学表达。难度越高 → 目标越小 → 预期哈希次数越大。

In [ ]:
def expected_hashes(target: int) -> float:
    return (2 ** 256) / target

def expected_time(target: int, hashrate: float) -> float:
    return expected_hashes(target) / hashrate

gen_target = bits_to_target(0x1d00ffff)
cur_target = difficulty_to_target(70_000_000_000_000)

def fmt_time(s):
    if s < 60: return f"{s:.1f}s"
    if s < 3600: return f"{s/60:.0f}m"
    if s < 86400: return f"{s/3600:.1f}h"
    if s < 86400*30: return f"{s/86400:.0f}d"
    return f"{s/86400/365:,.0f} years"

print(f"{'Miner':<22} | {'Genesis era':>14} | {'Current era':>18}")
print("-" * 60)
for name, hr in [
    ("Your MacBook (CPU)", 10_000),
    ("MacBook (GPU)", 3_000_000_000),
    ("1x Antminer S21", 200_000_000_000_000),
    ("Global hashrate", 600_000_000_000_000_000_000),
]:
    t_gen = expected_time(gen_target, hr)
    t_cur = expected_time(cur_target, hr)
    print(f"{name:<22} | {fmt_time(t_gen):>14} | {fmt_time(t_cur):>18}")

print("
💀 Even global hashrate needs ~10min for current difficulty")
print("💀 Your MacBook would need ~10^17 years")


## 2.4 Nonce 空间问题

区块头的 nonce 只有 **32 位**（0 ~ 4,294,967,295）。

现代 ASIC 做到 200 TH/s，每秒 2×10^14 次哈希。
**4.3×10^9 / 2×10^14 = 0.00002 秒** — nonce 就没了。

所以矿工不能只改 nonce。他们还会改 coinbase 交易的 extraNonce 字段、调整时间戳，来扩展搜索空间。

这也是矿池 Stratum 协议要给矿工分配不同 nonce 范围的原因。

## 2.5 挖矿是泊松过程

每次哈希独立同分布，成功概率 p = target/2^256（极低）。

→ 区块到达间隔服从 **指数分布**
→ 每天出块数服从 **泊松分布**

这意味着即使难度稳定，出块时间也会有显著波动。有时 1 分钟出 3 个块，有时 1 小时不出块——**都正常**。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n_blocks = 10000
target_time = 600  # seconds (10 minutes)
times = np.random.exponential(target_time, n_blocks)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(times / 60, bins=50, edgecolor="black", alpha=0.7, color="orange")
axes[0].axvline(x=10, color="red", linestyle="--", linewidth=2, label="Target = 10min")
axes[0].set_xlabel("Block time (minutes)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Block Time Distribution (Exponential)")
axes[0].legend()
axes[0].set_xlim(0, 60)

# Daily block count
daily = [sum(1 for t in times[i:i+144] if sum(times[i:i+144]) <= 86400) 
         for i in range(0, n_blocks - 144, 144)]
axes[1].hist(daily, bins=30, edgecolor="black", alpha=0.7)
axes[1].axvline(x=144, color="red", linestyle="--", linewidth=2, label="Expected = 144")
axes[1].set_xlabel("Blocks per day")
axes[1].set_ylabel("Days")
axes[1].set_title("Daily Block Count (Poisson)")
axes[1].legend()

plt.tight_layout()
plt.savefig("../data/poisson_mining.png", dpi=100, bbox_inches="tight")
print("✅ Saved to data/poisson_mining.png")
plt.close("all")


## 小结

✅ nBits = 压缩浮点存储难度目标
✅ Difficulty = 相对值 (genesis_target / current_target)
✅ E[hashes] = 2^256 / target — 难度越高越难
✅ Nonce 空间不够 → extraNonce 扩展
✅ Poisson 过程 → 区块时间自然波动，正常现象

**下一课**: 区块结构和 Merkle 树——区块里到底装了什么。